# SOC Triage Classifier — `train/triage.ipynb`
COS30018 Option D · SOC Triage ML pipeline

Classical-ML triage stage: classifies CICIDS2017 flows into 7 categories and
emits a `triage_output` dict, consumed downstream via
`session_state["triage_output"]` by a Detection LlmAgent.

**Dataset:** `dhoogla/cicids2017` Kaggle **Version 2** (no-metadata parquet,
one file per attack day/category). Attach it as a Kaggle notebook input, then
run top-to-bottom.

**Stop gates (by design — report, don't edit the gate):**
1. Schema gate — exactly `EXPECTED_N_FEATURES = 77` feature columns.
2. Label gate — every raw label must map to one of the 7 categories.
   **Heartbleed does not map and halts the notebook**; it is excluded only
   after you explicitly set `ACKNOWLEDGE_HEARTBLEED = True`.
3. Leakage guard — no flow id / IP / port / timestamp / label-derived column
   may survive into the feature set, and nothing fits before the split.

# 1. Setup

In [ ]:
!pip -q install xgboost shap >/dev/null 2>&1

import os, glob, json, time, random, hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn, xgboost, shap

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = "/kaggle/input"    # parquet files are discovered recursively under this
OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

EXPECTED_N_FEATURES = 77
LABEL_COL = "Label"
CATEGORIES = ["benign", "dos_ddos", "port_scan", "brute_force",
              "web_attack", "botnet", "infiltration"]
MODEL_VERSION = "triage-xgb-1.0.0"

print("numpy", np.__version__, "| pandas", pd.__version__,
      "| sklearn", sklearn.__version__, "| xgboost", xgboost.__version__,
      "| shap", shap.__version__)
print("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR, "| SEED:", SEED)

# 2. Load Dataset

In [ ]:
parquet_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.parquet"), recursive=True))
if not parquet_files:
    raise SystemExit(f"STOP: no .parquet files under {DATA_DIR} — "
                     "attach dhoogla/cicids2017 (Version 2, no-metadata) as a notebook input.")

frames = []
for p in parquet_files:
    f = pd.read_parquet(p)
    f.columns = [str(c).strip() for c in f.columns]
    print(f"{os.path.basename(p):<45} {f.shape}")
    frames.append(f)
df = pd.concat(frames, ignore_index=True)
del frames
print("\ncombined:", df.shape)

In [ ]:
print("dtypes:")
print(df.dtypes.value_counts(), "\n")

if LABEL_COL not in df.columns:
    raise SystemExit(f"STOP: no '{LABEL_COL}' column. Columns: {list(df.columns)}")
print("raw label counts:")
print(df[LABEL_COL].value_counts())

# 3. Schema Gate

In [ ]:
feature_cols = [c for c in df.columns if c != LABEL_COL]
if len(feature_cols) != EXPECTED_N_FEATURES:
    for i, c in enumerate(df.columns):
        print(f"{i:>3} {c}")
    raise SystemExit(f"STOP: {len(feature_cols)} feature columns != {EXPECTED_N_FEATURES}. "
                     "Report the full column list above before continuing.")
print(f"schema gate OK: {len(feature_cols)} feature columns")

# 4. Data Cleaning
Duplicates and constant columns dropped, Inf converted to NaN. No imputation yet — that is fit post-split, on train only.

In [ ]:
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"duplicate rows dropped: {n0 - len(df):,}  ({n0:,} -> {len(df):,})")

In [ ]:
constant_cols = [c for c in feature_cols if df[c].nunique(dropna=False) <= 1]
df = df.drop(columns=constant_cols)
feature_cols = [c for c in feature_cols if c not in constant_cols]
print(f"constant columns dropped: {len(constant_cols)} {constant_cols}")
print(f"feature columns remaining: {len(feature_cols)}")

In [ ]:
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)
nan_per_col = df[feature_cols].isna().sum()
n_nan_cols = int((nan_per_col > 0).sum())
print(f"NaN cells after Inf->NaN: {int(nan_per_col.sum()):,} across {n_nan_cols} columns")
if n_nan_cols:
    print(nan_per_col[nan_per_col > 0].to_string())
else:
    print("(no NaN present — the post-split imputer below stays as a safety net)")

# 5. Label Engineering
Keyword matching survives CICIDS2017's dash/space/case variants; `web_attack` must match before `brute_force` ("Web Attack - Brute Force").

In [ ]:
def to_category(label):
    s = " ".join(str(label).strip().lower()
                 .replace("\u2013", "-").replace("\u2014", "-").split())
    if s == "benign":                             return "benign"
    if "web attack" in s or "web-attack" in s:    return "web_attack"
    if "brute" in s or "patator" in s:            return "brute_force"
    if "portscan" in s or "port scan" in s:       return "port_scan"
    if "ddos" in s or "dos" in s:                 return "dos_ddos"
    if "bot" in s:                                return "botnet"
    if "infiltration" in s:                       return "infiltration"
    return None

label_map = {raw: to_category(raw) for raw in df[LABEL_COL].unique()}
for raw, cat in sorted(label_map.items(), key=lambda kv: str(kv[1])):
    print(f"{str(raw)!r:<42} -> {cat}")

In [ ]:
ACKNOWLEDGE_HEARTBLEED = False   # set True after reading the STOP message, then re-run this cell

unmapped = [raw for raw, cat in label_map.items() if cat is None]
heartbleed_labels = [r for r in unmapped if "heartbleed" in str(r).lower()]
other_unmapped = [r for r in unmapped if r not in heartbleed_labels]
if other_unmapped:
    raise SystemExit(f"STOP: unmapped non-Heartbleed labels {other_unmapped} — "
                     "extend the taxonomy, never bucket or drop silently.")

hb_mask = df[LABEL_COL].isin(heartbleed_labels)
n_heartbleed = int(hb_mask.sum())
print(f"Heartbleed rows found: {n_heartbleed:,}")

if n_heartbleed and not ACKNOWLEDGE_HEARTBLEED:
    raise SystemExit(
        "STOP: 'Heartbleed' does not map to any of the 7 project categories.\n"
        f"{n_heartbleed} rows are quarantined, NOT silently dropped.\n"
        "Per the project spec it stays outside the taxonomy.\n"
        "Set ACKNOWLEDGE_HEARTBLEED = True above and re-run this cell to exclude the rows;\n"
        "the exclusion is recorded in results.json and the model card."
    )

heartbleed_df = df[hb_mask].copy()
df = df[~hb_mask].reset_index(drop=True)
df["category"] = df[LABEL_COL].map(label_map)
assert df["category"].notna().all()
print(f"excluded {n_heartbleed} Heartbleed rows (kept in heartbleed_df); remaining: {len(df):,}")
print("\ncategory distribution:")
print(df["category"].value_counts().reindex(CATEGORIES, fill_value=0))

# 6. Leakage Guard

In [ ]:
LEAKAGE_PATTERNS = ["flow id", "src ip", "source ip", "dst ip", "destination ip",
                    "src port", "source port", "dst port", "destination port",
                    "timestamp", "label"]
leaked = [c for c in feature_cols if any(p in c.lower() for p in LEAKAGE_PATTERNS)]
if leaked:
    raise SystemExit(f"STOP: potential leakage columns in features: {leaked}")
assert LABEL_COL not in feature_cols and "category" not in feature_cols

FITTING_ALLOWED = False   # flipped True only by the split cell; every fit asserts on it
print(f"leakage guard OK: no id/ip/port/timestamp/label columns among {len(feature_cols)} features")

# 7. Train/Val/Test Split
Stratified 70/15/15, seed 42. Labels encoded to the canonical `CATEGORIES` order.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[feature_cols]
y = df["category"].map({c: i for i, c in enumerate(CATEGORIES)}).to_numpy()

X_trainval, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_train_val, test_size=0.15 / 0.85, stratify=y_train_val, random_state=SEED)

FITTING_ALLOWED = True
for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    counts = np.bincount(yy, minlength=len(CATEGORIES))
    print(f"{name:<5} {len(yy):>9,} rows | "
          + " ".join(f"{c}:{n:,}" for c, n in zip(CATEGORIES, counts)))

# 8. Imputation & Scaling
Fit on **train only**, post-split. Saved as `preprocessor.joblib` for the serving path.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

assert FITTING_ALLOWED, "split must run before any fitting"

preprocessor = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
Xtr = preprocessor.fit_transform(X_train).astype(np.float32)   # fit on train ONLY
Xva = preprocessor.transform(X_val).astype(np.float32)
Xte = preprocessor.transform(X_test).astype(np.float32)

joblib.dump(preprocessor, os.path.join(OUT_DIR, "preprocessor.joblib"))
print("preprocessor fit on train only — saved preprocessor.joblib")
print("shapes:", Xtr.shape, Xva.shape, Xte.shape)

# 9. Baselines
Logistic Regression and Random Forest, evaluated on the **validation** split (the test split is touched once, in Evaluation).

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

results = {}

def evaluate(name, model, X_eval, y_true):
    pred = model.predict(X_eval)
    macro = f1_score(y_true, pred, average="macro")
    results[name] = {
        "macro_f1": round(float(macro), 4),
        "per_class": classification_report(y_true, pred, labels=range(len(CATEGORIES)),
                                           target_names=CATEGORIES, output_dict=True,
                                           zero_division=0),
    }
    print(f"=== {name} — macro-F1 {macro:.4f} ===")
    print(classification_report(y_true, pred, labels=range(len(CATEGORIES)),
                                target_names=CATEGORIES, digits=3, zero_division=0))
    print("confusion matrix (rows=true, cols=pred):")
    print(pd.DataFrame(confusion_matrix(y_true, pred, labels=range(len(CATEGORIES))),
                       index=CATEGORIES, columns=CATEGORIES))
    return pred

print("evaluate() ready — results collected into `results`")

In [ ]:
from sklearn.linear_model import LogisticRegression

assert FITTING_ALLOWED
t0 = time.perf_counter()
logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
logreg.fit(Xtr, y_train)
print(f"LogisticRegression trained in {time.perf_counter() - t0:.0f}s")
evaluate("logreg_val", logreg, Xva, y_val);

In [ ]:
from sklearn.ensemble import RandomForestClassifier

assert FITTING_ALLOWED
t0 = time.perf_counter()
rf = RandomForestClassifier(n_estimators=100, max_depth=25, min_samples_leaf=5,
                            class_weight="balanced", n_jobs=-1, random_state=SEED)
rf.fit(Xtr, y_train)
print(f"RandomForest trained in {time.perf_counter() - t0:.0f}s")
evaluate("rf_val", rf, Xva, y_val);

# 10. XGBoost
Small grid tuned on a stratified subsample of train with early stopping on the
**validation** split, then the best config is refit on train+val at the best
iteration count. Uses GPU automatically if the Kaggle accelerator is on.

In [ ]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
import shutil

DEVICE = "cuda" if shutil.which("nvidia-smi") else "cpu"

TUNE_MAX_ROWS = 300_000
if len(Xtr) > TUNE_MAX_ROWS:
    Xtune, _, ytune, _ = train_test_split(Xtr, y_train, train_size=TUNE_MAX_ROWS,
                                          stratify=y_train, random_state=SEED)
else:
    Xtune, ytune = Xtr, y_train
w_tune = compute_sample_weight("balanced", ytune)
print(f"device: {DEVICE} | tuning on {len(ytune):,} of {len(y_train):,} train rows")

param_grid = [{"max_depth": d, "learning_rate": lr} for d in (6, 8, 10) for lr in (0.1, 0.3)]
tuning = []
for params in param_grid:
    assert FITTING_ALLOWED
    m = XGBClassifier(n_estimators=300, early_stopping_rounds=20, tree_method="hist",
                      device=DEVICE, eval_metric="mlogloss", random_state=SEED,
                      n_jobs=-1, **params)
    m.fit(Xtune, ytune, sample_weight=w_tune, eval_set=[(Xva, y_val)], verbose=False)
    macro = f1_score(y_val, m.predict(Xva), average="macro")
    tuning.append({**params, "best_iteration": int(m.best_iteration),
                   "val_macro_f1": round(float(macro), 4)})
    print(tuning[-1])

best = max(tuning, key=lambda t: t["val_macro_f1"])
print("\nbest config:", best)

In [ ]:
final_params = {"max_depth": best["max_depth"], "learning_rate": best["learning_rate"]}
n_rounds = best["best_iteration"] + 1

Xtrv = np.vstack([Xtr, Xva])
ytrv = np.concatenate([y_train, y_val])
w_trv = compute_sample_weight("balanced", ytrv)

t0 = time.perf_counter()
final_model = XGBClassifier(n_estimators=n_rounds, tree_method="hist", device=DEVICE,
                            eval_metric="mlogloss", random_state=SEED, n_jobs=-1,
                            **final_params)
final_model.fit(Xtrv, ytrv, sample_weight=w_trv, verbose=False)
final_model.set_params(device="cpu")   # inference on CPU — matches SOC deployment
print(f"final XGBoost: {n_rounds} rounds, params {final_params}, "
      f"fit on train+val ({len(ytrv):,} rows) in {time.perf_counter() - t0:.0f}s")

# 11. Evaluation
All three models reported once on the held-out **test** split.

In [ ]:
pred_lr = evaluate("logreg_test", logreg, Xte, y_test)
pred_rf = evaluate("rf_test", rf, Xte, y_test)
pred_xgb = evaluate("xgboost_test", final_model, Xte, y_test)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_test, pred_xgb, display_labels=CATEGORIES,
                                        xticks_rotation=45, ax=ax, colorbar=False,
                                        values_format=",")
ax.set_title("XGBoost — test confusion matrix")
plt.tight_layout()
plt.show()
print("rows = true category, cols = predicted")

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

proba_xgb = final_model.predict_proba(Xte)
fig, ax = plt.subplots(figsize=(8, 6))
ap_per_class = {}
for i, cat in enumerate(CATEGORIES):
    y_bin = (y_test == i).astype(int)
    prec, rec, _ = precision_recall_curve(y_bin, proba_xgb[:, i])
    ap = average_precision_score(y_bin, proba_xgb[:, i])
    ap_per_class[cat] = round(float(ap), 4)
    ax.plot(rec, prec, label=f"{cat} (AP={ap:.3f})")
ax.set_xlabel("recall"); ax.set_ylabel("precision")
ax.set_title("XGBoost — one-vs-rest PR curves (test)")
ax.legend(loc="lower left")
plt.show()

results["average_precision"] = ap_per_class
print(ap_per_class)

In [ ]:
t0 = time.perf_counter()
_ = final_model.predict_proba(Xte)
batch_ms_per_flow = (time.perf_counter() - t0) / len(Xte) * 1000

one = Xte[:1]
_ = final_model.predict_proba(one)   # warm-up
t0 = time.perf_counter()
for _i in range(200):
    _ = final_model.predict_proba(one)
single_flow_ms = (time.perf_counter() - t0) / 200 * 1000

results["latency"] = {"batch_ms_per_flow": round(batch_ms_per_flow, 4),
                      "single_flow_ms": round(single_flow_ms, 3)}
print(results["latency"])

# 12. SHAP
Global importance from mean |SHAP| over a 2,000-row test sample and all classes, plus one local explanation for a predicted attack.

In [ ]:
rng = np.random.default_rng(SEED)
shap_idx = rng.choice(len(Xte), size=min(2000, len(Xte)), replace=False)
X_shap = Xte[shap_idx]

explainer = shap.TreeExplainer(final_model)
sv = explainer.shap_values(X_shap)
sv = np.stack(sv, axis=-1) if isinstance(sv, list) else sv   # -> (n_rows, n_features, n_classes)

global_imp = pd.Series(np.abs(sv).mean(axis=(0, 2)), index=feature_cols).sort_values()
global_imp.tail(20).plot.barh(figsize=(8, 7),
                              title="SHAP global importance — mean |value| over sample & classes")
plt.tight_layout()
plt.show()
print(global_imp.tail(10)[::-1].round(4).to_string())

In [ ]:
pred_sample = final_model.predict(X_shap)
local_i = int(np.argmax(pred_sample != 0)) if (pred_sample != 0).any() else 0
cls = int(pred_sample[local_i])

contrib = pd.Series(sv[local_i, :, cls], index=feature_cols)
top = contrib.reindex(contrib.abs().sort_values().tail(10).index)
top.plot.barh(figsize=(8, 5),
              title=f"Local SHAP — sample row {local_i}, predicted '{CATEGORIES[cls]}'")
plt.tight_layout()
plt.show()
print(f"true={CATEGORIES[int(y_test[shap_idx][local_i])]} | predicted={CATEGORIES[cls]}")
print(top[::-1].round(4).to_string())

# 13. Output Contract
`triage_output` schema consumed by the Detection LlmAgent via
`session_state["triage_output"]`.

**Triage outputs a ranked, scored candidate list. It does not classify an
event as malicious or determine final severity — that decision belongs to
the Detection LlmAgent, which reasons over this output plus log/behavior/CTI
context.**

- `predicted_category` — the top-ranked candidate, not a verdict.
- `top_k_categories` — all 7 categories with their probabilities, sorted
  descending, so Detection can see competing hypotheses that a bare argmax
  would hide.
- `requires_review` — routing signal for Detection: `True` when `confidence`
  is below `CONFIDENCE_REVIEW_THRESHOLD` (0.6), or when the top-2
  probabilities are within `MARGIN_REVIEW_THRESHOLD` (0.15) of each other
  (an ambiguous call between two categories).
- `severity_score` — a priority score for triage ranking (category base
  severity × confidence), **not** a final severity classification.
- `entities` stays null for CICIDS2017 benchmark runs — populated only
  during live capture.

In [ ]:
SEVERITY_BASE = {
    "benign": 0.0, "port_scan": 0.35, "brute_force": 0.55, "web_attack": 0.70,
    "dos_ddos": 0.80, "botnet": 0.85, "infiltration": 0.95,
}
CONFIDENCE_REVIEW_THRESHOLD = 0.6
MARGIN_REVIEW_THRESHOLD = 0.15

def build_triage_output(event_id, proba_row):
    """Score and rank candidate categories for Detection to adjudicate — not a verdict."""
    order = np.argsort(proba_row)[::-1]
    top_k_categories = [
        {"category": CATEGORIES[i], "probability": round(float(proba_row[i]), 4)}
        for i in order
    ]
    top_i = int(order[0])
    predicted_category = CATEGORIES[top_i]
    confidence = float(proba_row[top_i])
    margin = float(proba_row[order[0]] - proba_row[order[1]])
    requires_review = bool(
        confidence < CONFIDENCE_REVIEW_THRESHOLD or margin < MARGIN_REVIEW_THRESHOLD
    )
    return {
        "event_id": str(event_id),
        "predicted_category": predicted_category,
        "confidence": round(confidence, 4),
        "top_k_categories": top_k_categories,
        "requires_review": requires_review,
        # priority score for triage ranking, not a final severity classification
        "severity_score": round(SEVERITY_BASE[predicted_category] * confidence, 4),
        "entities": {"src_ip": None, "dst_ip": None, "dst_port": None, "protocol": None},
        "model_version": MODEL_VERSION,
    }

# one confident example (largest top-2 margin) + one ambiguous example (smallest margin)
sorted_proba = np.sort(proba_xgb, axis=1)[:, ::-1]
margins = sorted_proba[:, 0] - sorted_proba[:, 1]
confident_row = int(np.argmax(margins))
ambiguous_row = int(np.argmin(margins))

demo = [
    build_triage_output(f"evt-{confident_row:06d}", proba_xgb[confident_row]),
    build_triage_output(f"evt-{ambiguous_row:06d}", proba_xgb[ambiguous_row]),
]
print(json.dumps(demo, indent=2))
print("requires_review:", [d["requires_review"] for d in demo])

# 14. Export
`model.joblib`, `preprocessor.joblib` (saved in §8), `results.json`, `model_card.md` — all under `OUT_DIR`.

In [ ]:
joblib.dump(final_model, os.path.join(OUT_DIR, "model.joblib"))

h = hashlib.sha256()
h.update(pd.util.hash_pandas_object(df[sorted(feature_cols)], index=False).values.tobytes())
data_hash = h.hexdigest()[:16]

results.update({
    "model_version": MODEL_VERSION,
    "seed": SEED,
    "source_dataset": "dhoogla/cicids2017 (version 2, no-metadata)",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "data_hash": data_hash,
    "n_rows_after_cleaning": int(len(df)),
    "heartbleed_rows_excluded": n_heartbleed,
    "constant_cols_dropped": constant_cols,
    "n_features": len(feature_cols),
    "feature_cols": feature_cols,
    "categories": CATEGORIES,
    "severity_base": SEVERITY_BASE,
    "split": {"train": int(len(y_train)), "val": int(len(y_val)), "test": int(len(y_test))},
    "xgb_tuning_log": tuning,
    "xgb_best": best,
})
with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2, default=str)

print("saved model.joblib + results.json")
print("test macro-F1 — logreg:", results["logreg_test"]["macro_f1"],
      "| rf:", results["rf_test"]["macro_f1"],
      "| xgboost:", results["xgboost_test"]["macro_f1"])

In [ ]:
model_card = f"""# Model Card — SOC Triage Classifier `{MODEL_VERSION}`

## Overview
XGBoost multiclass triage model for COS30018 Option D. Produces ranked,
scored candidates over 7 categories ({", ".join(CATEGORIES)}) for a
Detection LlmAgent to adjudicate. Feeds
`session_state["triage_output"]` to that Detection LlmAgent downstream.

## Data
- Source: dhoogla/cicids2017, Kaggle Version 2 (no-metadata parquet)
- Rows after cleaning: {len(df):,} (duplicates + constant columns removed, Inf -> NaN)
- Heartbleed rows excluded (outside the 7-class taxonomy, per spec): {n_heartbleed}
- Features: {len(feature_cols)} | data hash: `{data_hash}`
- Split: stratified 70/15/15, seed {SEED}

## Method
- Preprocessing: median imputation + standard scaling, fit on train only
- Baselines: Logistic Regression, Random Forest (class-weight balanced)
- Final model: XGBoost (hist), tuned on validation with early stopping,
  refit on train+val at {best["best_iteration"] + 1} rounds,
  params {final_params}, balanced sample weights

## Test results (macro F1)
| model | macro F1 |
|---|---|
| Logistic Regression | {results["logreg_test"]["macro_f1"]} |
| Random Forest | {results["rf_test"]["macro_f1"]} |
| **XGBoost** | **{results["xgboost_test"]["macro_f1"]}** |

Latency: {results["latency"]["single_flow_ms"]} ms/flow single,
{results["latency"]["batch_ms_per_flow"]} ms/flow batched (CPU).
Per-class metrics, PR curves and confusion matrices: see `results.json` and notebook.

## Kaggle-notebook flaws avoided
1. **Leakage via bootstrap resampling** — no resampling at all; duplicates are
   removed *before* the split so no identical flow lands in both train and
   test. Imbalance is handled with class/sample weights instead.
2. **Scaler fit on the test set** — imputer and scaler are fit on the training
   split only, then applied to val/test.
3. **CV / tuning on the test set** — hyperparameters were selected on the
   validation split; the test split was used exactly once, for final reporting.
4. **Missing random state** — seed {SEED} fixed for numpy, all splits and
   every model.

## Output contract
`triage_output` is a ranked, scored candidate list, not a final verdict:
event_id, predicted_category (top-ranked candidate), confidence,
top_k_categories (all 7 categories + probabilities, sorted descending),
requires_review (True when confidence < {CONFIDENCE_REVIEW_THRESHOLD} or the
top-2 margin < {MARGIN_REVIEW_THRESHOLD} — the routing signal to Detection),
severity_score (= category base severity x confidence, a priority score for
ranking, not a final severity classification), entities (null for
CICIDS2017 benchmark runs — populated only during live capture),
model_version. Triage scores and prioritizes; classifying an event as
malicious and determining final severity is the Detection LlmAgent's job.

## Limitations
- Trained on 2017 benchmark traffic; distribution shift expected on live networks.
- `infiltration` has very few samples — its metrics are noisy.
- Heartbleed is not covered by the taxonomy and is not detected by this model.
"""

with open(os.path.join(OUT_DIR, "model_card.md"), "w") as f:
    f.write(model_card)
print(model_card)

## Artifacts

- `model.joblib` — final XGBoost model
- `preprocessor.joblib` — median imputer + standard scaler (fit on train only)
- `results.json` — all metrics, tuning log, split sizes, data hash
- `model_card.md` — method, results, flaws avoided, limitations

Next step: wire `build_triage_output()` + these artifacts into the serving
path that fills `session_state["triage_output"]` for the Detection LlmAgent.